In [1]:
import dspy 
from config.settings import settings


In [2]:

settings.model_dump()

{'llm_model': 'openai/gemma-4-31B-it',
 'llm_api_base': 'https://openrouter.ai/api/v1',
 'llm_api_key': 'Oj_63y_gBe8qlO4IDb0cR0w+FsrsOEr7bOCkPqxNrZ<Pg+sreD',
 'llm_timeout': 30,
 'llm_retry_attempts': 5,
 'llm_max_concurrent': 50}

In [14]:
import pandas as pd

dsm_df = pd.read_json("books/dsm-tree/dsm5-final.jsonl", lines=True)
dsm_df

,page_number,content,book_name,tree,llm
0,26,### **An Uncertain Diagnosis**\n\n### When you...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
1,2,### **Also from James Morrison**\n\n### Diagno...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2,3,# **DSM-5-TR** **®**\n\n# **Made Easy**\n\n# *...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
3,4,EPUB Edition ISBN: 9781462551361\n\nCopyright ...,DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
4,5,"### *For Mary, always my sine qua non*",DSM5_ME,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
...,...,...,...,...,...
2213,1368,"of prolonged grief disorder, 324 of PTSD, 311 ...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2214,1343,"Kleptomania, 539 – 541 Korea, prevalence of\n\...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2215,1373,"Systemic racism, 17 – 18\n\nTachypnea, and neu...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it
2216,1320,"Cannabis, 129 , 147 , 159 , 471 , 503 . *See a...",DSM5_TR,"{'type': 'root', 'heading': None, 'content': '...",openai/gemma-4-31B-it


In [29]:
import pandas as pd
import uuid

# Example tree (your input)
page_tree = {
    'type': 'root',
    'heading': None,
    'content': '',
    'children': [
        {'type': 'paragraph', 'heading': None, 'content': 'OK, so these pointers aren’t exactly iron-clad. Remember, they’re straws, not steel.', 'children': []},
        {'type': 'heading', 'heading': 'Essential Features of Psychotic Disorder Due to Another', 'content': 'Essential Features of Psychotic Disorder Due to Another', 'children': [
            {'type': 'heading', 'heading': 'Medical Condition', 'content': 'Medical Condition', 'children': [
                {'type': 'paragraph', 'heading': None, 'content': 'Through physiological means, a medical condition appears to have caused an illness that features obvious hallucinations or delusions.', 'children': []},
                {'type': 'heading', 'heading': 'The Fine Print', 'content': 'The Fine Print', 'children': [
                    {'type': 'paragraph', 'heading': None, 'content': 'For pointers on deciding when a physical condition may have caused a mental disorder, see the sidebar above .', 'children': []}
                ]},
                {'type': 'paragraph', 'heading': None, 'content': 'The D’s: • Distress or disability (work/academic, social, or personal impairment) • Differential diagnosis ( delirium, another mental disorder, substance-induced psychotic disorder, schizophrenia and its cousins, delusional disorder)', 'children': []},
                {'type': 'heading', 'heading': 'Coding Notes', 'content': 'Coding Notes', 'children': [
                    {'type': 'paragraph', 'heading': None, 'content': 'In recording the diagnosis, use the name of the responsible medical condition, and list first the medical condition, with its code number.', 'children': []},
                    {'type': 'paragraph', 'heading': None, 'content': 'Code, based on the predominant symptoms:\n\nF06.2 With delusions F06.0 With hallucinations', 'children': []},
                    {'type': 'paragraph', 'heading': None, 'content': 'You may specify severity, though you don’t have to ( p. 74 ).', 'children': []}
                ]}
            ]}
        ]},
        {'type': 'case_study', 'heading': 'Rodrigo Chavez', 'content': 'Rodrigo Chavez', 'children': [
            {'type': 'paragraph', 'heading': None, 'content': 'Since retiring from teaching at age 65, Rodrigo Chavez spends most of his time sitting alone in his room. Sometimes he plays the acoustic guitar; once or twice he’s shot targets at the rifle range. True to his lifelong custom, he never drinks. Other than his immediate family, he has few social contacts. “My cigarettes are my best friends,” he says during the forensic examination.', 'children': []},
            {'type': 'paragraph', 'heading': None, 'content': 'When Rodrigo is nearly 70, an inoperable carcinoma of the lung is diagnosed. After a course of palliative radiotherapy, he declines further treatment and settles down in his apartment to die. Four months later, he notices right-sided headaches that will sometimes awaken him in the middle of the night. Because the doctors have told him he is terminally ill, he doesn’t seek further medical attention.', 'children': []},
            {'type': 'paragraph', 'heading': None, 'content': 'Then, he begins to associate the headaches with natural gas, which he can smell coming out of the ventilator duct in his bathroom. When he calls to report the problem to Mrs. Riordan, his landlady, she sends around the building’s handyman, who can find nothing wrong. But both the odors and his headaches have increased; Rodrigo recalls that, weeks earlier, Mrs. Riordan went out several times to watch while', 'children': []}
        ]}
    ]
}

#  Recursive function to assign unique IDs and mark leaf/parent
def assign_ids(node, book_name="DSM5", parent_id=None, page_number=1, nodes_list=None):
    if nodes_list is None:
        nodes_list = []

    node_id = str(uuid.uuid4())  # unique ID
    # Determine if node is a leaf or has children
    node_type_in_tree = 'leaf' if not node.get('children') else 'parent'

    node_record = {
        'node_id': node_id,
        'parent_id': parent_id,
        'type': node.get('type'),
        'heading': node.get('heading'),
        'content': node.get('content'),
        'page': page_number,
        'node_type_in_tree': node_type_in_tree,
        'book_name': book_name
    }
    nodes_list.append(node_record)

    # Recursively assign IDs to children
    for child in node.get('children', []):
        assign_ids(child, book_name=book_name, parent_id=node_id, page_number=page_number, nodes_list=nodes_list)

    return nodes_list

# Assign IDs and get list of nodes
nodes_with_ids = assign_ids(page_tree, page_number=1)

# Convert to DataFrame
df = pd.DataFrame(nodes_with_ids)

# Display the DataFrame
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name
0,57774cec-9c8a-4007-a17f-30531290f8ea,NaN,root,NaN,,1,parent,DSM5
1,9169ee97-9385-4b98-a108-75f57256a127,57774cec-9c8a-4007-a17f-30531290f8ea,paragraph,NaN,"OK, so these pointers aren’t exactly iron-clad...",1,leaf,DSM5
2,870238da-6c60-46d1-923e-c9bb38696a06,57774cec-9c8a-4007-a17f-30531290f8ea,heading,Essential Features of Psychotic Disorder Due t...,Essential Features of Psychotic Disorder Due t...,1,parent,DSM5
3,f627a60b-193c-4df1-96b5-67b791d23187,870238da-6c60-46d1-923e-c9bb38696a06,heading,Medical Condition,Medical Condition,1,parent,DSM5
4,fc4c6e20-167d-4fe1-897b-12532a35bd1b,f627a60b-193c-4df1-96b5-67b791d23187,paragraph,NaN,"Through physiological means, a medical conditi...",1,leaf,DSM5
5,c5178dd8-54f6-4a37-904b-0d170ca99d9c,f627a60b-193c-4df1-96b5-67b791d23187,heading,The Fine Print,The Fine Print,1,parent,DSM5
6,85f2fa40-1d68-4b42-a99a-c5670cce2cee,c5178dd8-54f6-4a37-904b-0d170ca99d9c,paragraph,NaN,For pointers on deciding when a physical condi...,1,leaf,DSM5
7,76a0f183-828b-4030-860c-c90fb90a92f3,f627a60b-193c-4df1-96b5-67b791d23187,paragraph,NaN,The D’s: • Distress or disability (work/academ...,1,leaf,DSM5
8,e71d00ad-c63f-4ef3-9dcf-5e8b95cb143e,f627a60b-193c-4df1-96b5-67b791d23187,heading,Coding Notes,Coding Notes,1,parent,DSM5
9,4512b437-e170-46ec-b5e8-bc6907789646,e71d00ad-c63f-4ef3-9dcf-5e8b95cb143e,paragraph,NaN,"In recording the diagnosis, use the name of th...",1,leaf,DSM5


In [30]:
nodes_with_ids = []
for i, j in dsm_df.iterrows():
    nodes_with_ids = assign_ids(j['tree'], page_number=j['page_number'], nodes_list=nodes_with_ids,book_name=j['book_name'])
    df = pd.DataFrame(nodes_with_ids)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name
0,d775b317-2373-4c28-8da7-c03fe4ed7272,NaN,root,NaN,,26,parent,DSM5_ME
1,c9bccfea-3e21-429f-8152-6adc879798bd,d775b317-2373-4c28-8da7-c03fe4ed7272,heading,An Uncertain Diagnosis,An Uncertain Diagnosis,26,parent,DSM5_ME
2,0b39a0c1-a8ff-4b3b-bc85-b10e09671ba5,c9bccfea-3e21-429f-8152-6adc879798bd,paragraph,NaN,When you’re not sure whether a diagnosis is co...,26,leaf,DSM5_ME
3,5a913b1b-edce-4abf-95d4-e9a6f6e6310a,c9bccfea-3e21-429f-8152-6adc879798bd,paragraph,NaN,What about a patient who comes very close to m...,26,leaf,DSM5_ME
4,63979f91-73cf-4bc9-8a42-f4e0534be255,d775b317-2373-4c28-8da7-c03fe4ed7272,heading,Indicating Severity of a Disorder,Indicating Severity of a Disorder,26,leaf,DSM5_ME
...,...,...,...,...,...,...,...,...
27363,ff78531e-8c34-43e5-b2ab-b33132b09707,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,"coding of, 822",1371,leaf,DSM5_TR
27364,f06a1f66-a8af-453d-ba23-9910a7c9aa76,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,"contextual information in DSM-5-TR, 27",1371,leaf,DSM5_TR
27365,86428e6a-1381-48fc-89b8-bf5d36a6085e,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,"depressive disorders and, 190 , 195 , 199 , 20...",1371,leaf,DSM5_TR
27366,ad03ad83-e0d6-4ee7-8bc1-7421acf2ad82,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,depressive episodes with short-duration hypoma...,1371,leaf,DSM5_TR


In [43]:
# Define chapter start pages based on the TOC you provided
chapter_starts = [
    (0, "Also Available"),
    (3, "Title Page"),
    (4, "Copyright"),
    (5, "Dedication"),
    (6, "About the Author"),
    (7, "Acknowledgments"),
    (9, "Contents"),
    (12, "Frequently Needed Tables"),
    (13, "Introduction"),
    (34, "Chapter 1: Neurodevelopmental Disorders"),
    (64, "Chapter 2: Schizophrenia Spectrum and Other Psychotic Disorders"),  # adjust as per actual TOC
    (117, "Chapter 3: Mood Disorders"),
    (164, "Chapter 4: Anxiety Disorders"),
    (190, "Chapter 5: Obsessive–Compulsive and Related Disorders"),
    (207, "Chapter 6: Trauma- and Stressor-Related Disorders"),
    (228, "Chapter 7: Dissociative Disorders"),
    (241, "Chapter 8: Somatic Symptom and Related Disorders"),
    (267, "Chapter 9: Feeding and Eating Disorders"),
    (283, "Chapter 10: Elimination Disorders"),
    (335, "Chapter 12: Sexual Dysfunctions"),
    (356, "Chapter 13: Gender Dysphoria"),
    (363, "Chapter 14: Disruptive, Impulse-Control, and Conduct Disorders"),
    (377, "Chapter 15: Substance-Related and Addictive Disorders"),
    (436, "Chapter 16: Cognitive Disorders"),
    (490, "Chapter 17: Personality Disorders"),
    (524, "Chapter 18: Paraphilic Disorders"),
    (548, "Chapter 19: Other Factors That May Need Clinical Attention"),
    (563, "Chapter 20: Patients and Diagnoses"),
    (600, "Appendix: Essential Tables"),
    (608, "Index"),
    (631, "About Guilford Press"),
    (632, "Discover Related Guilford Books"),
]

# Sort chapters by page (just in case)
chapter_starts = sorted(chapter_starts, key=lambda x: x[0])

# Function to map page to chapter
def get_chapter_for_page_for_dsm5_DE(page_number):
    chapter_name = None
    for i, (start_page, name) in enumerate(chapter_starts):
        # If page is before the next chapter start or last chapter
        if i + 1 < len(chapter_starts):
            next_start, _ = chapter_starts[i + 1]
            if start_page <= page_number < next_start:
                chapter_name = name
                break
        else:  # Last chapter
            if page_number >= start_page:
                chapter_name = name
                break
    return chapter_name

# Example usage
pages_to_test = [10, 35, 120, 365, 500, 605]
for page in pages_to_test:
    print(f"Page {page} → {get_chapter_for_page_for_dsm5_DE(page)}")

Page 10 → Contents
Page 35 → Chapter 1: Neurodevelopmental Disorders
Page 120 → Chapter 3: Mood Disorders
Page 365 → Chapter 14: Disruptive, Impulse-Control, and Conduct Disorders
Page 500 → Chapter 17: Personality Disorders
Page 605 → Appendix: Essential Tables


In [44]:
# Updated chapter start pages based on your new TOC
chapter_starts = [
    (0, "Cover Page"),
    (6, "Title Page"),
    (7, "Copyright Page"),
    (8, "Contents"),
    (10, "DSM-5-TR Chairs and Review Groups"),
    (24, "DSM-5 Task Force and Work Groups"),
    (35, "Preface to DSM-5-TR"),
    (37, "Preface to DSM-5"),
    (41, "DSM-5-TR Classification"),
    (92, "Section I DSM-5 Basics"),
    (94, "Introduction"),
    (113, "Use of the Manual"),
    (123, "Cautionary Statement for Forensic Use of DSM-5"),
    (126, "Section II Diagnostic Criteria and Codes"),
    (130, "Neurodevelopmental Disorders"),  # you suggested 130 for unclear
    (207, "Schizophrenia Spectrum and Other Psychotic Disorders"),
    (254, "Bipolar and Related Disorders"),
    (301, "Depressive Disorders"),
    (349, "Anxiety Disorders"),
    (407, "Obsessive-Compulsive and Related Disorders"),
    (447, "Trauma- and Stressor-Related Disorders"),
    (490, "Dissociative Disorders"),
    (515, "Somatic Symptom and Related Disorders"),
    (543, "Feeding and Eating Disorders"),
    (576, "Elimination Disorders"),
    (586, "Sleep-Wake Disorders"),
    (671, "Sexual Dysfunctions"),
    (712, "Gender Dysphoria"),
    (726, "Disruptive, Impulse-Control, and Conduct Disorders"),
    (753, "Substance-Related and Addictive Disorders"),
    (903, "Neurocognitive Disorders"),
    (979, "Personality Disorders"),
    (1035, "Paraphilic Disorders"),
    (1064, "Other Mental Disorders and Additional Codes"),
    (1068, "Medication-Induced Movement Disorders and Other Adverse Effects"),
    (1100, "Other Conditions That May Be a Focus of Clinical Attention"),
    (1104, "Section III Emerging Measures and Models"),
    (1107, "Assessment Measures"),
    (1124, "Culture and Psychiatric Diagnosis"),
    (1145, "Alternative DSM-5 Model for Personality Disorders"),
    (1167, "Conditions for Further Study"),
    (1197, "Appendix"),
    (1307, "Index")
]

# Sort just in case
chapter_starts = sorted(chapter_starts, key=lambda x: x[0])

# Function to map page to chapter
def get_chapter_for_page_for_dsm5_TR(page_number):
    chapter_name = "Unclear Section"  # default if no match
    for i, (start_page, name) in enumerate(chapter_starts):
        if i + 1 < len(chapter_starts):
            next_start, _ = chapter_starts[i + 1]
            if start_page <= page_number < next_start:
                chapter_name = name
                break
        else:  # Last chapter
            if page_number >= start_page:
                chapter_name = name
                break
    return chapter_name

# Example usage
pages_to_test = [5, 40, 150, 365, 580, 720, 1000]
for page in pages_to_test:
    print(f"Page {page} → {get_chapter_for_page_for_dsm5_TR(page)}")

Page 5 → Cover Page
Page 40 → Preface to DSM-5
Page 150 → Neurodevelopmental Disorders
Page 365 → Anxiety Disorders
Page 580 → Elimination Disorders
Page 720 → Gender Dysphoria
Page 1000 → Personality Disorders


In [45]:
def get_chapter_for_page(book_name, page_number):
    if(book_name=="DSM5_TR"):
        return get_chapter_for_page_for_dsm5_TR(page_number)
    else:
        return get_chapter_for_page_for_dsm5_DE(page_number)

get_chapter_for_page("DSM_DE", 1) 

'Cover Page'

In [47]:
df['section'] = df.apply(lambda row: get_chapter_for_page(row['book_name'], row['page']), axis=1)
df

,node_id,parent_id,type,heading,content,page,node_type_in_tree,book_name,section
0,d775b317-2373-4c28-8da7-c03fe4ed7272,NaN,root,NaN,,26,parent,DSM5_ME,DSM-5 Task Force and Work Groups
1,c9bccfea-3e21-429f-8152-6adc879798bd,d775b317-2373-4c28-8da7-c03fe4ed7272,heading,An Uncertain Diagnosis,An Uncertain Diagnosis,26,parent,DSM5_ME,DSM-5 Task Force and Work Groups
2,0b39a0c1-a8ff-4b3b-bc85-b10e09671ba5,c9bccfea-3e21-429f-8152-6adc879798bd,paragraph,NaN,When you’re not sure whether a diagnosis is co...,26,leaf,DSM5_ME,DSM-5 Task Force and Work Groups
3,5a913b1b-edce-4abf-95d4-e9a6f6e6310a,c9bccfea-3e21-429f-8152-6adc879798bd,paragraph,NaN,What about a patient who comes very close to m...,26,leaf,DSM5_ME,DSM-5 Task Force and Work Groups
4,63979f91-73cf-4bc9-8a42-f4e0534be255,d775b317-2373-4c28-8da7-c03fe4ed7272,heading,Indicating Severity of a Disorder,Indicating Severity of a Disorder,26,leaf,DSM5_ME,DSM-5 Task Force and Work Groups
...,...,...,...,...,...,...,...,...,...
27363,ff78531e-8c34-43e5-b2ab-b33132b09707,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,"coding of, 822",1371,leaf,DSM5_TR,Index
27364,f06a1f66-a8af-453d-ba23-9910a7c9aa76,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,"contextual information in DSM-5-TR, 27",1371,leaf,DSM5_TR,Index
27365,86428e6a-1381-48fc-89b8-bf5d36a6085e,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,"depressive disorders and, 190 , 195 , 199 , 20...",1371,leaf,DSM5_TR,Index
27366,ad03ad83-e0d6-4ee7-8bc1-7421acf2ad82,1b407a29-b0fc-4f35-b60d-aa34b4e0b78b,list_item,NaN,depressive episodes with short-duration hypoma...,1371,leaf,DSM5_TR,Index


In [49]:
df.to_json("books/dsm-tree/tree_with_IDs.json", orient="records", lines=True)

In [ ]:
df.groupby("")

In [ ]:


lm = dspy.LM(
    model=settings.llm_model,
    api_base=settings.llm_api_base,
    api_key=settings.llm_api_key,
    timeout=settings.llm_timeout,
)
dspy.settings.configure(lm=lm, temperature=0.0)


In [3]:
from signatures.entity_extraction import DisorderEntityExtractor
from signatures.entity_example import case_study_para2_entities, case_study_para1_entities, example_entities
from dspy.teleprompt import BootstrapFewShot

trainset = [case_study_para2_entities, case_study_para1_entities, example_entities]
cot = dspy.ChainOfThought(DisorderEntityExtractor)
optimizer = BootstrapFewShot()
compiled_cot = optimizer.compile(
    cot,
    trainset=trainset
)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00,  7.73it/s]

Bootstrapped 3 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.
